# BAML demo

## Load the client and the types

In [ ]:
from baml_client import b
from baml_client.types import CodeAction, FinalAnswer

## Try to make a call to the LLM provider with the defined prompt

In [ ]:
from src.engine.tools.created.count_ifc_elements import count_ifc_elements

# Define the variables for the test
user_input = "how many doors are there in this house?"
model_path = "/Users/sylvainhellin/GitHub/4_phd/bim-qas/src/experiment/bim_models/duplex/arc.ifc"
available_tools = """
def count_ifc_elements(model_path: str, ifc_type: str, name_pattern: str = None, property_filter: dict = None) -> int:
    '''Count the number of instances of a specific IFC entity type in a BIM model. Args: model_path (str): Path to the IFC model file ifc_type (str): The IFC entity type to count (e.g., "IfcWindow", "IfcDoor", "IfcWall") name_pattern (str, optional): Pattern to filter elements by name (case-insensitive substring match) property_filter (dict, optional): Dictionary of property name-value pairs to filter elements. Properties can be: - Direct attributes of the element (e.g., 'Name', 'GlobalId', 'id') - Properties in the element's info dictionary - Properties within property sets (e.g., 'FireRating' in 'Pset_DoorCommon') Note: This function works with IFC models exported from Revit and similar BIM authoring software that include property sets like PSet_Revit_*. Returns: int: The count of elements matching the specified criteria Raises: Exception: If the model cannot be loaded or if the specified entity type doesn't exist'''
    """

# Make an API call to the LLM provider
result = b.BIMQAS(
    user_input=user_input,
    available_tools=available_tools,
    previous_attempts=None,
    model_path=model_path,
)

print(f"Type of the result: {type(result)}\n")
print(f"Result:\n{result.model_dump_json(indent=2)}\n")

## Basic control flow with the type system

In [ ]:
if isinstance(result, CodeAction):
    print("The Agent wants to execute some python code.")
    print(f"Executing following code: {result.python_code[:100]}...\n")
    # NOTE: here, I cannot access result.answer

elif isinstance(result, FinalAnswer):
    print("The Agent has a final answer.")
    print(f"Final Answer: {result.answer}")
    # NOTE: here, I cannot access result.python_code


## More refined loops

In [ ]:
previous_attempts = None
for i in range(10):
    print(f"Starting iteration Nr. {i}")

    result = b.BIMQAS(
        user_input=user_input,
        available_tools=available_tools,
        previous_attempts=previous_attempts,
        model_path=model_path,
    )

    print("\n#####\n")

    # Deal with final answer
    if isinstance(result, FinalAnswer):
        print(f"\nAgent is done with it's task. Final Answer:\n{result.answer}\n")
        break

    # Deal with loop execution
    elif isinstance(result, CodeAction):
        print("\nThe Agent wants to execute some python code.")
        print(f"Code to execute: {result.python_code[:100]}...")

        # Update the previous attempts (context)
        if previous_attempts is None:
            previous_attempts = ""
        previous_attempts += f"\nThought {i+1}:\n{result.thoughts}\n"
        previous_attempts += f"\nPython Code {i+1}:\n{result.python_code}\n"
        # previous_attempts += f"\nResult {i+1}:\n{eval(result.python_code)}\n"
        previous_attempts += "\nResult {i+1}:\nTotal number of doors 14.\n" # NOTE: here hardcoded because of week logic for parsing code
        previous_attempts += "\n#####\n"
        continue


